In [2]:
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_score, KFold
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np
import joblib

# Load data
df = pd.read_csv("autofocus_training_final 1.3_2500.csv")  # your CSV name
feature_cols = ['Z_prev', 'variance_ratio', 'Variance_sq',
                'is_improving', 'direction', 'dVariance_abs']
X = df[feature_cols]
y = df['target_step']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale (required for SVR)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Parameter grid – adjust as needed
param_grid = {
    'C':       [1, 10, 100, 1000],
    'epsilon': [0.1, 1, 10, 50, 100],
    'gamma':   ['scale', 'auto', 0.01, 0.1, 1]
}

svr = SVR(kernel='rbf')

# Run Grid Search
grid = GridSearchCV(svr, param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=1)
grid.fit(X_train_scaled, y_train)

# --- FIX: Define best_svr FIRST before using it ---
best_svr = grid.best_estimator_

# 5-fold CV on the scaled training data using the best estimator
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2_scores = cross_val_score(best_svr, X_train_scaled, y_train, cv=cv, scoring='r2')

print("\n5‑fold CV R² scores for SVR:", cv_r2_scores)
print(f"Mean CV R²: {cv_r2_scores.mean():.4f} (+/- {cv_r2_scores.std():.4f})")

print("Best parameters:", grid.best_params_)
print("Best CV R²:", grid.best_score_)

# Evaluate on test set with best estimator
y_pred = best_svr.predict(X_test_scaled)
mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print("\n====== Tuned SVR Test Metrics ======")
print(f"MAE  : {mae:.2f} steps")
print(f"RMSE : {rmse:.2f} steps")
print(f"R²   : {r2:.4f}")

# Save the tuned SVR and scaler
joblib.dump(best_svr, 'svr_autofocus_tunedv55.pkl')
joblib.dump(scaler, 'scaler_svr_tunedv55.pkl')
print("\nModel and scaler successfully saved!")

Fitting 5 folds for each of 100 candidates, totalling 500 fits

5‑fold CV R² scores for SVR: [0.97165974 0.9668122  0.95915894 0.95407899 0.97406517]
Mean CV R²: 0.9652 (+/- 0.0075)
Best parameters: {'C': 1000, 'epsilon': 100, 'gamma': 'scale'}
Best CV R²: 0.9653290145727329

====== Tuned SVR Test Metrics ======
MAE  : 224.87 steps
RMSE : 291.78 steps
R²   : 0.9696

Model and scaler successfully saved!
